# Scenario Dreamer Assets (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/scenario-dreamer-decision-layer/blob/main/notebooks/scenario_dreamer_assets_colab.ipynb)

This notebook prepares the **persistent Drive-backed baseline substrate**:
- sync this repo,
- clone and pin upstream Scenario Dreamer,
- bind canonical repo asset paths to Google Drive,
- optionally download the pretrained CtRL-Sim checkpoint,
- verify the checkpoint and Scenario Dreamer environment-pack locations.


In [1]:
# Step 1: Sync this repo into the Colab runtime
import os
import subprocess
import sys

REPO_URL = 'https://github.com/achyutmorang/scenario-dreamer-decision-layer.git'
REPO_DIR = '/content/scenario-dreamer-decision-layer'

if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '-b', 'main', REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', REPO_DIR], check=True)
for p in (REPO_DIR, os.path.join(REPO_DIR, 'src')):
    if p not in sys.path:
        sys.path.insert(0, p)

print('repo_rev:', subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD'], text=True).strip())


repo_rev: 093dc4e


In [2]:
# Suggested defaults: Drive-backed persistent asset layout.
%env SCENARIO_DREAMER_DRIVE_ROOT=/content/drive/MyDrive/scenario_dreamer_research
%env SCENARIO_DREAMER_DOWNLOAD_CHECKPOINT=1
%env SCENARIO_DREAMER_DOWNLOAD_ENVS=0



env: SCENARIO_DREAMER_DRIVE_ROOT=/content/drive/MyDrive/scenario_dreamer_research
env: SCENARIO_DREAMER_DOWNLOAD_CHECKPOINT=1
env: SCENARIO_DREAMER_DOWNLOAD_ENVS=0


In [3]:
# Step 2: Mount Drive and inspect the default persistent layout
import json
import os
from pathlib import Path

from google.colab import drive
from scenario_dreamer_decision_layer.colab import default_drive_layout

drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT = os.environ['SCENARIO_DREAMER_DRIVE_ROOT']
layout = {k: str(v) for k, v in default_drive_layout(DRIVE_ROOT).items()}
print(json.dumps(layout, indent=2, sort_keys=True))


Mounted at /content/drive
{
  "assets_root": "/content/drive/MyDrive/scenario_dreamer_research/scenario_dreamer_decision_layer/assets",
  "base": "/content/drive/MyDrive/scenario_dreamer_research/scenario_dreamer_decision_layer",
  "datasets_root": "/content/drive/MyDrive/scenario_dreamer_research/scenario_dreamer_decision_layer/assets/datasets",
  "env_jsons_dir": "/content/drive/MyDrive/scenario_dreamer_research/scenario_dreamer_decision_layer/assets/scenario_dreamer_waymo_200m_jsons",
  "env_pickles_dir": "/content/drive/MyDrive/scenario_dreamer_research/scenario_dreamer_decision_layer/assets/scenario_dreamer_waymo_200m_pickles",
  "results_root": "/content/drive/MyDrive/scenario_dreamer_research/scenario_dreamer_decision_layer/results/runs",
  "scratch_root": "/content/drive/MyDrive/scenario_dreamer_research/scenario_dreamer_decision_layer/assets/scenario-dreamer"
}


In [4]:
# Step 3: Clone/pin upstream Scenario Dreamer, bind canonical repo asset paths to Drive, and seed the released env pack
import json
import os
import subprocess
import sys

from scenario_dreamer_decision_layer.colab import bind_drive_layout, inspect_bound_layout, seed_env_pack_from_upstream

subprocess.run([sys.executable, 'scripts/bootstrap_baseline.py', '--clone-upstream', '--write-lock'], check=True)
binding = bind_drive_layout(os.environ['SCENARIO_DREAMER_DRIVE_ROOT'])
print('binding:')
print(json.dumps(binding, indent=2, sort_keys=True))
print('inspection:')
print(json.dumps(inspect_bound_layout(), indent=2, sort_keys=True))
seed_payload = seed_env_pack_from_upstream()
print('seed_env_pack:')
print(json.dumps(seed_payload, indent=2, sort_keys=True))



binding:
{
  "bindings": {
    "datasets_root": {
      "actual": "/content/drive/MyDrive/scenario_dreamer_research/scenario_dreamer_decision_layer/assets/datasets",
      "canonical": "/content/scenario-dreamer-decision-layer/artifacts/assets/datasets",
      "mode": "symlink",
      "status": "bound"
    },
    "env_jsons_dir": {
      "actual": "/content/drive/MyDrive/scenario_dreamer_research/scenario_dreamer_decision_layer/assets/scenario_dreamer_waymo_200m_jsons",
      "canonical": "/content/scenario-dreamer-decision-layer/artifacts/assets/scenario_dreamer_waymo_200m_jsons",
      "mode": "symlink",
      "status": "bound"
    },
    "env_pickles_dir": {
      "actual": "/content/drive/MyDrive/scenario_dreamer_research/scenario_dreamer_decision_layer/assets/scenario_dreamer_waymo_200m_pickles",
      "canonical": "/content/scenario-dreamer-decision-layer/artifacts/assets/scenario_dreamer_waymo_200m_pickles",
      "mode": "symlink",
      "status": "bound"
    },
    "scratch_ro

In [5]:
# Step 4: Optional checkpoint and environment-pack download into Drive-backed canonical asset paths
import os
import subprocess
import sys

DOWNLOAD_CHECKPOINT = os.environ.get('SCENARIO_DREAMER_DOWNLOAD_CHECKPOINT', '0').strip().lower() in {'1', 'true', 'yes', 'y', 'on'}
DOWNLOAD_ENVS = os.environ.get('SCENARIO_DREAMER_DOWNLOAD_ENVS', '0').strip().lower() in {'1', 'true', 'yes', 'y', 'on'}
print('DOWNLOAD_CHECKPOINT:', DOWNLOAD_CHECKPOINT)
print('DOWNLOAD_ENVS:', DOWNLOAD_ENVS)
if DOWNLOAD_CHECKPOINT or DOWNLOAD_ENVS:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'gdown'], check=True)
    modes = []
    if DOWNLOAD_CHECKPOINT:
        modes.append('checkpoint')
    if DOWNLOAD_ENVS:
        modes.append('envs')
    for mode in modes:
        print(f'--- downloading mode={mode} ---')
        proc = subprocess.run(
            [sys.executable, 'scripts/download_assets.py', '--mode', mode, '--download'],
            text=True,
            capture_output=True,
            check=False,
        )
        if proc.stdout:
            print(proc.stdout)
        if proc.stderr:
            print(f'[step4_{mode}_stderr]')
            print(proc.stderr)
        if proc.returncode != 0:
            raise RuntimeError(f'Asset download or normalization failed for mode={mode}. Inspect the JSON payload above.')
else:
    print('Skipping asset downloads. The 75 pre-generated Scenario Dreamer environment files should already be seeded from the upstream repo in Step 3.')



DOWNLOAD_CHECKPOINT: True
DOWNLOAD_ENVS: False
--- downloading mode=checkpoint ---
{
  "after": {
    "checkpoint": {
      "actual_sha256": "8ed2d3a0546a06907f797224492c44b38013ae804af1de0fe9991814d12d0062",
      "exists": true,
      "expected_sha256": "8ed2d3a0546a06907f797224492c44b38013ae804af1de0fe9991814d12d0062",
      "google_drive_url": "https://drive.google.com/drive/folders/1vw84BCfqqolY4DTl3YXOZwfY6nYnCmNo?usp=sharing",
      "path": "/content/scenario-dreamer-decision-layer/artifacts/assets/scenario-dreamer/checkpoints/ctrl_sim_waymo_1M_steps/last.ckpt"
    },
    "simulation_envs": {
      "jsons_dir": "/content/scenario-dreamer-decision-layer/artifacts/assets/scenario_dreamer_waymo_200m_jsons",
      "jsons_exists": true,
      "num_jsons": 75,
      "num_pickles": 75,
      "pickles_dir": "/content/scenario-dreamer-decision-layer/artifacts/assets/scenario_dreamer_waymo_200m_pickles",
      "pickles_exists": true,
      "source_url": "https://drive.google.com/drive/fol

In [6]:
# Step 5: Verify current asset state
import json
import subprocess
import sys

proc = subprocess.run(
    [sys.executable, 'scripts/download_assets.py', '--mode', 'all'],
    text=True,
    capture_output=True,
    check=False,
)
if proc.stdout:
    print(proc.stdout)
if proc.stderr:
    print('[step5_verify_stderr]')
    print(proc.stderr)
verification_payload = json.loads(proc.stdout)
errors = verification_payload.get('verification_errors', [])
print('verification_summary:')
print(json.dumps({
    'checkpoint_exists': verification_payload['after']['checkpoint']['exists'],
    'checkpoint_path': verification_payload['after']['checkpoint']['path'],
    'num_pickles': verification_payload['after']['simulation_envs']['num_pickles'],
    'num_jsons': verification_payload['after']['simulation_envs']['num_jsons'],
    'verification_errors': errors,
}, indent=2, sort_keys=True))
if proc.returncode != 0:
    raise RuntimeError('Assets are not ready yet. Resolve the verification errors printed above before running the baseline notebook.')



{
  "after": {
    "checkpoint": {
      "actual_sha256": "8ed2d3a0546a06907f797224492c44b38013ae804af1de0fe9991814d12d0062",
      "exists": true,
      "expected_sha256": "8ed2d3a0546a06907f797224492c44b38013ae804af1de0fe9991814d12d0062",
      "google_drive_url": "https://drive.google.com/drive/folders/1vw84BCfqqolY4DTl3YXOZwfY6nYnCmNo?usp=sharing",
      "path": "/content/scenario-dreamer-decision-layer/artifacts/assets/scenario-dreamer/checkpoints/ctrl_sim_waymo_1M_steps/last.ckpt"
    },
    "simulation_envs": {
      "jsons_dir": "/content/scenario-dreamer-decision-layer/artifacts/assets/scenario_dreamer_waymo_200m_jsons",
      "jsons_exists": true,
      "num_jsons": 75,
      "num_pickles": 75,
      "pickles_dir": "/content/scenario-dreamer-decision-layer/artifacts/assets/scenario_dreamer_waymo_200m_pickles",
      "pickles_exists": true,
      "source_url": "https://drive.google.com/drive/folders/13DSHf2UhrvguD7i7iYL5SfSDhgLcW_ja?usp=sharing"
    }
  },
  "before": {
    "c